# LightAutoML + FE


In [1]:
!pip install -q lightautoml

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.5/412.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.1/216.1 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.5/309.5 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.5/206.5 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 7.5 MB/s eta 0:00:00


In [2]:
from lightautoml.automl.presets.tabular_presets import TabularAutoML
from lightautoml.tasks import Task

'nlp' extra dependency package 'fasttext-numpy2' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.


In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
import keras

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

2026-03-18 14:17:38.550636: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773843459.011641      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773843459.142907      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773843460.097274      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773843460.097317      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773843460.097320      24 computation_placer.cc:177] computation placer alr

In [4]:
import os
import random
import warnings
import numpy as np, pandas as pd
from colorama import Fore, Style
from importlib.metadata import version
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

warnings.filterwarnings('ignore')
from itertools import combinations

In [5]:
def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    
seed_everything(seed=42)

## Load Data

In [6]:
TARGET = 'Churn'

In [7]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/test.csv")
orig = pd.read_csv('/kaggle/input/datasets/cdeotte/s6e3-original-dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv')

_tc = pd.to_numeric(orig.TotalCharges, errors='coerce')
_tc[_tc.isnull()] = orig[_tc.isnull()].MonthlyCharges * orig[_tc.isnull()].tenure
orig.TotalCharges = _tc
print("Train shape:", train.shape)
print("Test shape :", test.shape)

Train shape: (594194, 21)
Test shape : (254655, 20)


In [8]:
train[TARGET] = train[TARGET].map({'No': 0, 'Yes': 1}).astype(int)
orig[TARGET]  = orig[TARGET].map({'No': 0, 'Yes': 1}).astype(int)

## Preprocess Features

V8 - bao gồm các tính năng từ: https://www.kaggle.com/code/blamerx/s6e3-tabm-advanced-features-0-91898-cv/notebook

In [9]:
def pctrank_against(values, reference):
    """Calculate percentile rank of values against reference distribution."""
    ref_sorted = np.sort(reference)
    return (np.searchsorted(ref_sorted, values) / len(ref_sorted)).astype('float32')

def zscore_against(values, reference):
    """Calculate z-score of values relative to reference distribution."""
    mu, sigma = np.mean(reference), np.std(reference)
    return (np.zeros(len(values), dtype='float32') if sigma == 0
            else ((values - mu) / sigma).astype('float32'))


In [10]:
CATS = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]

In [11]:
NUMS = ['tenure', 'MonthlyCharges', 'TotalCharges']
NEW_NUMS = []

# 1. Frequency Encoding
print("[1/7] Frequency Encoding...")
for col in NUMS:
    freq = pd.concat([train[col], orig[col], test[col]]).value_counts(normalize=True)
    for df in [train, test, orig]:
        df[f'FREQ_{col}'] = df[col].map(freq).fillna(0).astype('float32')
    NEW_NUMS.append(f'FREQ_{col}')

# 2. Arithmetic Features
print("[2/7] Arithmetic Features...")
for df in [train, test, orig]:
    df['charges_deviation']      = (df['TotalCharges'] - df['tenure'] * df['MonthlyCharges']).astype('float32')
    df['monthly_to_total_ratio'] = (df['MonthlyCharges'] / (df['TotalCharges'] + 1)).astype('float32')
    df['avg_monthly_charges']    = (df['TotalCharges'] / (df['tenure'] + 1)).astype('float32')
NEW_NUMS += ['charges_deviation', 'monthly_to_total_ratio', 'avg_monthly_charges']

# 3. Service Counts
print("[3/7] Service Counts...")
SVC = ['PhoneService','MultipleLines','OnlineSecurity','OnlineBackup',
       'DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
for df in [train, test, orig]:
    df['service_count'] = (df[SVC] == 'Yes').sum(axis=1).astype('float32')
    df['has_internet']  = (df['InternetService'] != 'No').astype('float32')
    df['has_phone']     = (df['PhoneService'] == 'Yes').astype('float32')
NEW_NUMS += ['service_count', 'has_internet', 'has_phone']

# 4. ORIG_proba mapping (from original dataset)
print("[4/7] Target Probability Mapping...")
for col in CATS + NUMS:
    tmp   = orig.groupby(col)[TARGET].mean()
    _name = f"ORIG_proba_{col}"
    train = train.merge(tmp.rename(_name), on=col, how="left")
    test  = test.merge(tmp.rename(_name), on=col, how="left")
    for df in [train, test]:
        df[_name] = df[_name].fillna(0.5).astype('float32')
    NEW_NUMS.append(_name)

print(f"   Created {len(NEW_NUMS)} base features")


[1/7] Frequency Encoding...
[2/7] Arithmetic Features...
[3/7] Service Counts...
[4/7] Target Probability Mapping...
   Created 28 base features


In [12]:
# 5. Distribution Features
print("[5/7] Distribution Features...")

orig_ch_tc  = orig.loc[orig[TARGET] == 1, 'TotalCharges'].values
orig_nc_tc  = orig.loc[orig[TARGET] == 0, 'TotalCharges'].values
orig_tc     = orig['TotalCharges'].values
orig_is_mc  = orig.groupby('InternetService')['MonthlyCharges'].mean()

for df in [train, test]:
    tc = df['TotalCharges'].values
    df['pctrank_nonchurner_TC'] = pctrank_against(tc, orig_nc_tc)
    df['pctrank_churner_TC']    = pctrank_against(tc, orig_ch_tc)
    df['pctrank_orig_TC']       = pctrank_against(tc, orig_tc)
    df['zscore_churn_gap_TC']   = (np.abs(zscore_against(tc, orig_ch_tc)) -
                                   np.abs(zscore_against(tc, orig_nc_tc))).astype('float32')
    df['zscore_nonchurner_TC']  = zscore_against(tc, orig_nc_tc)
    df['pctrank_churn_gap_TC']  = (pctrank_against(tc, orig_ch_tc) -
                                   pctrank_against(tc, orig_nc_tc)).astype('float32')
    df['resid_IS_MC']           = (df['MonthlyCharges'] - df['InternetService'].map(orig_is_mc).fillna(0)).astype('float32')
    
    for cat_col, out_col in [('InternetService','cond_pctrank_IS_TC'), ('Contract','cond_pctrank_C_TC')]:
        vals = np.zeros(len(df), dtype='float32')
        for cv in orig[cat_col].unique():
            mask = df[cat_col] == cv
            ref  = orig.loc[orig[cat_col] == cv, 'TotalCharges'].values
            if len(ref) > 0 and mask.sum() > 0:
                vals[mask] = pctrank_against(df.loc[mask, 'TotalCharges'].values, ref)
        df[out_col] = vals

NEW_NUMS += [
    'pctrank_nonchurner_TC', 'zscore_churn_gap_TC', 'pctrank_churn_gap_TC',
    'resid_IS_MC', 'cond_pctrank_IS_TC', 'zscore_nonchurner_TC',
    'pctrank_orig_TC', 'pctrank_churner_TC', 'cond_pctrank_C_TC'
]

# Quantile distance features
for q_label, q_val in [('q25', 0.25), ('q50', 0.50), ('q75', 0.75)]:
    ch_q = np.quantile(orig_ch_tc, q_val)
    nc_q = np.quantile(orig_nc_tc, q_val)
    for df in [train, test]:
        df[f'dist_To_ch_{q_label}']   = np.abs(df['TotalCharges'] - ch_q).astype('float32')
        df[f'dist_To_nc_{q_label}']   = np.abs(df['TotalCharges'] - nc_q).astype('float32')
        df[f'qdist_gap_To_{q_label}'] = (df[f'dist_To_nc_{q_label}'] - df[f'dist_To_ch_{q_label}']).astype('float32')
NEW_NUMS += [
    'qdist_gap_To_q50','dist_To_ch_q50','dist_To_nc_q50',
    'dist_To_nc_q25','qdist_gap_To_q25',
    'dist_To_nc_q75','dist_To_ch_q75','qdist_gap_To_q75'
]

print(f"   Created distribution features")

[5/7] Distribution Features...
   Created distribution features


In [13]:
# 6. Digit Features
print("[6/7] Digit Features...")

DIGIT_FEATURES = [
    'tenure_first_digit','tenure_last_digit','tenure_second_digit',
    'tenure_mod10','tenure_mod12','tenure_num_digits',
    'tenure_is_multiple_10','tenure_rounded_10','tenure_dev_from_round10',
    'mc_first_digit','mc_last_digit','mc_second_digit',
    'mc_mod10','mc_mod100','mc_num_digits',
    'mc_is_multiple_10','mc_is_multiple_50',
    'mc_rounded_10','mc_fractional','mc_dev_from_round10',
    'tc_first_digit','tc_last_digit','tc_second_digit',
    'tc_mod10','tc_mod100','tc_num_digits',
    'tc_is_multiple_10','tc_is_multiple_100',
    'tc_rounded_100','tc_fractional','tc_dev_from_round100',
    'tenure_years','tenure_months_in_year','mc_per_digit','tc_per_digit'
]

for df in [train, test]:
    t_str  = df['tenure'].astype(str)
    df['tenure_first_digit']      = t_str.str[0].astype(int)
    df['tenure_last_digit']       = t_str.str[-1].astype(int)
    df['tenure_second_digit']     = t_str.apply(lambda x: int(x[1]) if len(x) > 1 else 0)
    df['tenure_mod10']            = df['tenure'] % 10
    df['tenure_mod12']            = df['tenure'] % 12
    df['tenure_num_digits']       = t_str.str.len()
    df['tenure_is_multiple_10']   = (df['tenure'] % 10 == 0).astype('float32')
    df['tenure_rounded_10']       = np.round(df['tenure'] / 10) * 10
    df['tenure_dev_from_round10'] = np.abs(df['tenure'] - df['tenure_rounded_10'])

    mc_str = df['MonthlyCharges'].astype(str).str.replace('.', '', regex=False)
    df['mc_first_digit']      = mc_str.str[0].astype(int)
    df['mc_last_digit']       = mc_str.str[-1].astype(int)
    df['mc_second_digit']     = mc_str.apply(lambda x: int(x[1]) if len(x) > 1 else 0)
    df['mc_mod10']            = np.floor(df['MonthlyCharges']) % 10
    df['mc_mod100']           = np.floor(df['MonthlyCharges']) % 100
    df['mc_num_digits']       = np.floor(df['MonthlyCharges']).astype(int).astype(str).str.len()
    df['mc_is_multiple_10']   = (np.floor(df['MonthlyCharges']) % 10 == 0).astype('float32')
    df['mc_is_multiple_50']   = (np.floor(df['MonthlyCharges']) % 50 == 0).astype('float32')
    df['mc_rounded_10']       = np.round(df['MonthlyCharges'] / 10) * 10
    df['mc_fractional']       = df['MonthlyCharges'] - np.floor(df['MonthlyCharges'])
    df['mc_dev_from_round10'] = np.abs(df['MonthlyCharges'] - df['mc_rounded_10'])

    tc_str = df['TotalCharges'].astype(str).str.replace('.', '', regex=False)
    df['tc_first_digit']       = tc_str.str[0].astype(int)
    df['tc_last_digit']        = tc_str.str[-1].astype(int)
    df['tc_second_digit']      = tc_str.apply(lambda x: int(x[1]) if len(x) > 1 else 0)
    df['tc_mod10']             = np.floor(df['TotalCharges']) % 10
    df['tc_mod100']            = np.floor(df['TotalCharges']) % 100
    df['tc_num_digits']        = np.floor(df['TotalCharges']).astype(int).astype(str).str.len()
    df['tc_is_multiple_10']    = (np.floor(df['TotalCharges']) % 10 == 0).astype('float32')
    df['tc_is_multiple_100']   = (np.floor(df['TotalCharges']) % 100 == 0).astype('float32')
    df['tc_rounded_100']       = np.round(df['TotalCharges'] / 100) * 100
    df['tc_fractional']        = df['TotalCharges'] - np.floor(df['TotalCharges'])
    df['tc_dev_from_round100'] = np.abs(df['TotalCharges'] - df['tc_rounded_100'])

    df['tenure_years']         = df['tenure'] // 12
    df['tenure_months_in_year']= df['tenure'] % 12
    df['mc_per_digit']         = df['MonthlyCharges'] / (df['mc_num_digits'] + 0.001)
    df['tc_per_digit']         = df['TotalCharges'] / (df['tc_num_digits'] + 0.001)

    for c in DIGIT_FEATURES:
        df[c] = df[c].astype('float32')

NEW_NUMS += DIGIT_FEATURES
print(f"   Created {len(DIGIT_FEATURES)} digit features")

[6/7] Digit Features...
   Created 35 digit features


In [14]:
# Categorical columns for n-gram interactions
TOP_CATS_FOR_NGRAM = [
    'Contract', 'InternetService', 'PaymentMethod',
    'OnlineSecurity', 'TechSupport', 'PaperlessBilling'
]

In [15]:
# 7. N-gram Categorical Interactions
print("[7/7] N-gram Features...")

BIGRAM_COLS, TRIGRAM_COLS = [], []

# Bigrams: all pairs from top-6 categorical columns
for c1, c2 in combinations(TOP_CATS_FOR_NGRAM, 2):
    col_name = f"BG_{c1}_{c2}"
    for df in [train, test]:
        df[col_name] = (df[c1].astype(str) + "_" + df[c2].astype(str))
    BIGRAM_COLS.append(col_name)

# Trigrams: all triples from top-4 categorical columns
TOP4 = TOP_CATS_FOR_NGRAM[:4]
for c1, c2, c3 in combinations(TOP4, 3):
    col_name = f"TG_{c1}_{c2}_{c3}"
    for df in [train, test]:
        df[col_name] = (df[c1].astype(str) + "_" + df[c2].astype(str) + "_" + df[c3].astype(str))
    TRIGRAM_COLS.append(col_name)

NGRAM_COLS = BIGRAM_COLS + TRIGRAM_COLS

# Numericals as categories for TE
NUM_AS_CAT = []
for col in NUMS:
    _new = f'CAT_{col}'
    NUM_AS_CAT.append(_new)
    for df in [train, test]:
        df[_new] = df[col].astype(str)

TE_COLUMNS = NUM_AS_CAT + CATS

print(f"   Created {len(NGRAM_COLS)} n-gram columns ({len(BIGRAM_COLS)} bi-grams + {len(TRIGRAM_COLS)} tri-grams)")
print(f"\n✅ Feature Engineering Complete!")
print(f"   Numerical features: {len(NUMS + NEW_NUMS)}")
print(f"   Categorical columns: {len(TE_COLUMNS)}")
print(f"   N-gram columns: {len(NGRAM_COLS)}")

[7/7] N-gram Features...
   Created 19 n-gram columns (15 bi-grams + 4 tri-grams)

✅ Feature Engineering Complete!
   Numerical features: 83
   Categorical columns: 19
   N-gram columns: 19


## Train K-Fold

In [16]:
import time

In [17]:
import gc

print("="*70)
print("TRAINING TAB TRANSFORMER WITH 10-FOLD CROSS-VALIDATION")
print("="*70)

N_FOLDS = 5
INNER_FOLDS = 5

skf       = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
skf_inner = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=42)

oof         = np.zeros(len(train))
pred        = np.zeros(len(test))
fold_scores = []
y_all       = train[TARGET].values

t0_all = time.time()
t0 = time.time()

for fold_i, (train_idx, val_idx) in enumerate(skf.split(train, y_all)):
    print(f"\n{'─'*50}")
    print(f"Fold {fold_i+1}/{N_FOLDS}")
    print(f"{'─'*50}")

    X_tr  = train.iloc[train_idx].reset_index(drop=True).copy()
    y_tr  = y_all[train_idx]
    X_val = train.iloc[val_idx].reset_index(drop=True).copy()
    y_val = y_all[val_idx]
    X_te  = test.copy()

    # ═══════════════════════════════════════════════════════════════
    # TARGET ENCODING (Leakage-Safe)
    # ═══════════════════════════════════════════════════════════════

    # Inner K-Fold TE for original categoricals (prevents leakage)
    te_feat_names = [f"TE1_{col}_mean" for col in TE_COLUMNS]
    for df in [X_tr, X_val, X_te]:
        for c in te_feat_names:
            df[c] = 0.0

    X_tr[TARGET] = y_tr
    for j, (in_tr, in_va) in enumerate(skf_inner.split(X_tr, y_tr)):
        X_tr2 = X_tr.iloc[in_tr]
        for col in TE_COLUMNS:
            tmp = X_tr2.groupby(col)[TARGET].mean().rename(f"TE1_{col}_mean")
            merged = X_tr.iloc[in_va][[col]].merge(tmp, on=col, how='left')
            X_tr.loc[X_tr.index[in_va], f"TE1_{col}_mean"] = merged[f"TE1_{col}_mean"].values

    for col in TE_COLUMNS:
        tmp = X_tr.groupby(col)[TARGET].mean().rename(f"TE1_{col}_mean")
        X_val[f"TE1_{col}_mean"] = X_val[[col]].merge(tmp, on=col, how='left')[f"TE1_{col}_mean"].values
        X_te[f"TE1_{col}_mean"]  = X_te[[col]].merge(tmp, on=col, how='left')[f"TE1_{col}_mean"].values
    X_tr.drop(columns=[TARGET], inplace=True)

    for df in [X_tr, X_val, X_te]:
        for c in te_feat_names:
            df[c] = df[c].fillna(0.5).astype('float32')

    # N-gram TE (full-fold mean - more stable for neural networks)
    ng_te_feat_names = [f"TE_ng_{col}" for col in NGRAM_COLS]
    X_tr[TARGET] = y_tr
    for col in NGRAM_COLS:
        ng_te = X_tr.groupby(col)[TARGET].mean()
        ng_n  = f"TE_ng_{col}"
        X_tr[ng_n]  = X_tr[col].map(ng_te).fillna(0.5).astype('float32')
        X_val[ng_n] = X_val[col].map(ng_te).fillna(0.5).astype('float32')
        X_te[ng_n]  = X_te[col].map(ng_te).fillna(0.5).astype('float32')
    X_tr.drop(columns=[TARGET], inplace=True)

    # ═══════════════════════════════════════════════════════════════
    # PREPARE DATA FOR TABM
    # ═══════════════════════════════════════════════════════════════

    ALL_NUMS_FINAL = NUMS + NEW_NUMS + te_feat_names + ng_te_feat_names
    ALL_CATS_FINAL = CATS

    if fold_i == 0:
        print(f"Numeric features: {len(ALL_NUMS_FINAL)}")
        print(f"Categorical features: {len(ALL_CATS_FINAL)}")

    # Ordinal encode categoricals
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    encoder.fit(X_tr[ALL_CATS_FINAL].astype(str))
    X_tr_cat  = encoder.transform(X_tr[ALL_CATS_FINAL].astype(str))
    X_val_cat = encoder.transform(X_val[ALL_CATS_FINAL].astype(str))
    X_te_cat  = encoder.transform(X_te[ALL_CATS_FINAL].astype(str))

    # Scale numericals
    for df in [X_tr, X_val, X_te]:
        df[ALL_NUMS_FINAL] = df[ALL_NUMS_FINAL].fillna(0).astype('float32')

    scaler    = StandardScaler()
    X_tr_num  = scaler.fit_transform(X_tr[ALL_NUMS_FINAL])
    X_val_num = scaler.transform(X_val[ALL_NUMS_FINAL])
    X_te_num  = scaler.transform(X_te[ALL_NUMS_FINAL])

    # Combine features
    ALL_COLS = ALL_NUMS_FINAL + ALL_CATS_FINAL
    X_tr_final  = pd.DataFrame(np.hstack([X_tr_num,  X_tr_cat]),  columns=ALL_COLS)
    X_val_final = pd.DataFrame(np.hstack([X_val_num, X_val_cat]), columns=ALL_COLS)
    X_te_final  = pd.DataFrame(np.hstack([X_te_num,  X_te_cat]),  columns=ALL_COLS)

    for c in ALL_CATS_FINAL:
        X_tr_final[c]  = X_tr_final[c].astype(int)
        X_val_final[c] = X_val_final[c].astype(int)
        X_te_final[c]  = X_te_final[c].astype(int)

    # ═══════════════════════════════════════════════════════════════
    # TRAIN LAMA
    # ═══════════════════════════════════════════════════════════════
    
    model = TabularAutoML(
        task=Task(name='binary', metric='auc', greater_is_better=True),
        timeout = 3600,
        reader_params = {'cv': 5, 'random_state': 42}
    )
    X_tr_final['churn'] = y_tr
    X_val_final['churn'] = y_val
    
    oof_tab_tf = model.fit_predict(
        X_tr_final,
        roles={'target': 'churn'},
        verbose=3,
        valid_data=X_val_final
    ).data.T[0]
    test_tab_tf = model.predict(X_te_final).data.T[0]
    

    val_probs  = oof_tab_tf
    oof[val_idx] = val_probs
    test_probs = test_tab_tf
    pred      += test_probs / N_FOLDS

    fold_auc = roc_auc_score(y_val, val_probs)
    fold_scores.append(fold_auc)

    print(f"Fold {fold_i+1} AUC: {fold_auc:.5f} | Time: {(time.time()-t0)/60:.1f} min")

    del model, X_tr_final, X_val_final, X_te_final
    gc.collect()

print(f"\n{'='*70}")
print("TRAINING COMPLETE!")
print("="*70)

TRAINING TAB TRANSFORMER WITH 10-FOLD CROSS-VALIDATION

──────────────────────────────────────────────────
Fold 1/5
──────────────────────────────────────────────────
Numeric features: 121
Categorical features: 16
[14:18:58] Stdout logging level is INFO3.
[14:18:58] Copying TaskTimer may affect the parent PipelineTimer, so copy will create new unlimited TaskTimer
[14:18:58] Task: binary

[14:18:58] Start automl preset with listed constraints:
[14:18:58] - time: 3600.00 seconds
[14:18:58] - CPU: 4 cores
[14:18:58] - memory: 16 GB

[14:18:58] Train data shape: (475355, 138)

[14:19:18] Feats was rejected during automatic roles guess: []
[14:19:19] Layer 1 train process start. Time left 3578.29 secs
[14:20:10] Linear model: C = 1e-05 score = 0.9139014527730032
[14:20:14] Linear model: C = 5e-05 score = 0.9152133611645668
[14:20:17] Linear model: C = 0.0001 score = 0.9155537906310065
[14:20:23] Linear model: C = 0.0005 score = 0.9160973391254744
[14:20:23] Linear model: C = 0.001 score = 0

In [18]:
print('OOF score: %.5f' % roc_auc_score(y_all, oof))

OOF score: 0.91884


## Evaluation and Submission

In [19]:
df_oof = pd.DataFrame({'Churn': oof})
df_oof.to_csv(f'oof.csv', index=False)

df_test = pd.DataFrame({'Churn': pred})
df_test.to_csv(f'test.csv',index=False)

print("Saved oof to file")

Saved oof to file


In [20]:
sub = pd.DataFrame({
    'id': test['id'],
    'Churn': pred
})

sub.to_csv('submission.csv', index=False)